# HW 9

Author: Evan Whitfield

Course: ST 554 - Spring 2026

Homework 9 - Using Spark MLlib

## Data 

This dataset include data for the estimation of obesity levels in individuals from the countries of Mexico, Peru and Colombia, based on their eating habits and physical condition.

The targer variable will be `NObeyesdad`, which is the obesity level of the subject.

Obesity levels are calculated based off of a person's BMI (Body Mass Index), which is a direct calcuation using the person's `height` and `weight`. Therefore, we will not be using both of these at the same time in our process for determining obesity level. We may use `height`, but will most likely not use `weight` as a predictor.

More info can be found at: https://archive.ics.uci.edu/dataset/544/estimation+of+obesity+levels+based+on+eating+habits+and+physical+condition

And:
https://www.sciencedirect.com/science/article/pii/S2352340919306985?via%3Dihub

In [156]:
import numpy as np
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.feature import SQLTransformer, StringIndexer, Binarizer, VectorAssembler
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.regression import LinearRegression, DecisionTreeRegressor, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

spark = SparkSession.builder.getOrCreate()

In [76]:
obs_data = pd.read_csv("Obesity Raw Data.csv")
obs_data.head()

,Gender,Age,Height,Weight,family_history_with_overweight,FAVC,FCVC,NCP,CAEC,SMOKE,CH2O,SCC,FAF,TUE,CALC,MTRANS,NObeyesdad
0,Female,21.0,1.62,64.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,0.0,1.0,no,Public_Transportation,Normal_Weight
1,Female,21.0,1.52,56.0,yes,no,3.0,3.0,Sometimes,yes,3.0,yes,3.0,0.0,Sometimes,Public_Transportation,Normal_Weight
2,Male,23.0,1.80,77.0,yes,no,2.0,3.0,Sometimes,no,2.0,no,2.0,1.0,Frequently,Public_Transportation,Normal_Weight
3,Male,27.0,1.80,87.0,no,no,3.0,3.0,Sometimes,no,2.0,no,2.0,0.0,Frequently,Walking,Overweight_Level_I
4,Male,22.0,1.78,89.8,no,no,2.0,1.0,Sometimes,no,2.0,no,0.0,0.0,Sometimes,Public_Transportation,Overweight_Level_II


In [77]:
obesity = spark.createDataFrame(obs_data)
obesity.show(5)

+------+----+------+------+------------------------------+----+----+---+---------+-----+----+---+---+---+----------+--------------------+-------------------+
|Gender| Age|Height|Weight|family_history_with_overweight|FAVC|FCVC|NCP|     CAEC|SMOKE|CH2O|SCC|FAF|TUE|      CALC|              MTRANS|         NObeyesdad|
+------+----+------+------+------------------------------+----+----+---+---------+-----+----+---+---+---+----------+--------------------+-------------------+
|Female|21.0|  1.62|  64.0|                           yes|  no| 2.0|3.0|Sometimes|   no| 2.0| no|0.0|1.0|        no|Public_Transporta...|      Normal_Weight|
|Female|21.0|  1.52|  56.0|                           yes|  no| 3.0|3.0|Sometimes|  yes| 3.0|yes|3.0|0.0| Sometimes|Public_Transporta...|      Normal_Weight|
|  Male|23.0|   1.8|  77.0|                           yes|  no| 2.0|3.0|Sometimes|   no| 2.0| no|2.0|1.0|Frequently|Public_Transporta...|      Normal_Weight|
|  Male|27.0|   1.8|  87.0|                         

In [174]:
indexer_obese = StringIndexer(
    inputCol="NObeyesdad",
    outputCol="Obese_Number"
)

indexerTrans = indexer_obese.fit(obesity) #now indexerTrans will have a .transform() method
indexerTrans.transform(obesity).show(30)

+------+----+------+------+------------------------------+----+----+---+----------+-----+----+---+---+---+----------+--------------------+-------------------+------------+
|Gender| Age|Height|Weight|family_history_with_overweight|FAVC|FCVC|NCP|      CAEC|SMOKE|CH2O|SCC|FAF|TUE|      CALC|              MTRANS|         NObeyesdad|Obese_Number|
+------+----+------+------+------------------------------+----+----+---+----------+-----+----+---+---+---+----------+--------------------+-------------------+------------+
|Female|21.0|  1.62|  64.0|                           yes|  no| 2.0|3.0| Sometimes|   no| 2.0| no|0.0|1.0|        no|Public_Transporta...|      Normal_Weight|         5.0|
|Female|21.0|  1.52|  56.0|                           yes|  no| 3.0|3.0| Sometimes|  yes| 3.0|yes|3.0|0.0| Sometimes|Public_Transporta...|      Normal_Weight|         5.0|
|  Male|23.0|   1.8|  77.0|                           yes|  no| 2.0|3.0| Sometimes|   no| 2.0| no|2.0|1.0|Frequently|Public_Transporta...|  

Thankfully, the obesity types were coded numerically as 0, 1, and 2. Therefore, we can binaryize based off the value of 2.5. Values below 2.5 will be coded as 0 = Obese or 1 = Not Obese.

Because I want to use family history as one of my predictors, I need to index the string column `family_history_of_overweight`.

To get all of the code to work properly, we will put it in a pipeline.

In [175]:
binarizer = Binarizer(
    threshold=2.5,
    inputCol="Obese_Number",
    outputCol="Obese_Indicator"
)

indexer_family = StringIndexer(
    inputCol="family_history_with_overweight",
    outputCol="Family_History_Indicator" #0 is yes, 1 is no
)

pipeline = Pipeline(stages=[indexer_obese, binarizer, indexer_family])

binarized_data = pipeline.fit(obesity).transform(obesity)

binarized_data.show()

+------+----+------+------+------------------------------+----+----+---+----------+-----+----+---+---+---+----------+--------------------+-------------------+------------+---------------+------------------------+
|Gender| Age|Height|Weight|family_history_with_overweight|FAVC|FCVC|NCP|      CAEC|SMOKE|CH2O|SCC|FAF|TUE|      CALC|              MTRANS|         NObeyesdad|Obese_Number|Obese_Indicator|Family_History_Indicator|
+------+----+------+------+------------------------------+----+----+---+----------+-----+----+---+---+---+----------+--------------------+-------------------+------------+---------------+------------------------+
|Female|21.0|  1.62|  64.0|                           yes|  no| 2.0|3.0| Sometimes|   no| 2.0| no|0.0|1.0|        no|Public_Transporta...|      Normal_Weight|         5.0|            1.0|                     0.0|
|Female|21.0|  1.52|  56.0|                           yes|  no| 3.0|3.0| Sometimes|  yes| 3.0|yes|3.0|0.0| Sometimes|Public_Transporta...|      Norm

Now that we have all of the columns created, we will use only the necessary columns.

In [80]:
sqlTrans = SQLTransformer(
    statement = """
                SELECT Age, Family_History_Indicator, CH2O, FAF,  ROUND(Height * 39.37, 4) as Height_Inches, Obese_Indicator as label FROM __THIS__
                """
)

In [81]:
assembler = VectorAssembler(inputCols = ["Age", "Family_History_Indicator", "CH2O", "FAF", "Height_Inches"], outputCol = "features", handleInvalid = 'keep')

assembler.transform(
    sqlTrans.transform(binarized_data)).show(30)

+----+------------------------+----+---+-------------+-----+--------------------+
| Age|Family_History_Indicator|CH2O|FAF|Height_Inches|label|            features|
+----+------------------------+----+---+-------------+-----+--------------------+
|21.0|                     0.0| 2.0|0.0|      63.7794|  1.0|[21.0,0.0,2.0,0.0...|
|21.0|                     0.0| 3.0|3.0|      59.8424|  1.0|[21.0,0.0,3.0,3.0...|
|23.0|                     0.0| 2.0|2.0|       70.866|  1.0|[23.0,0.0,2.0,2.0...|
|27.0|                     1.0| 2.0|2.0|       70.866|  1.0|[27.0,1.0,2.0,2.0...|
|22.0|                     1.0| 2.0|0.0|      70.0786|  1.0|[22.0,1.0,2.0,0.0...|
|29.0|                     1.0| 2.0|0.0|      63.7794|  1.0|[29.0,1.0,2.0,0.0...|
|23.0|                     0.0| 2.0|1.0|       59.055|  1.0|[23.0,0.0,2.0,1.0...|
|22.0|                     1.0| 2.0|3.0|      64.5668|  1.0|[22.0,1.0,2.0,3.0...|
|24.0|                     0.0| 2.0|1.0|      70.0786|  1.0|[24.0,0.0,2.0,1.0...|
|22.0|          

Now we have all of the necessary values as numeric in our dataset. We can now perform our fits to compare our models on predicting obesity.

`binarized_data` served as a temporary dataset for testing purposes. Now that we have finished all of the transforms to get our correct data, we can put it in a new Pipeline and fit it to our original dataset `obesity.'

*We will most likely build another Pipeline containing this and more when we are ready to determine our different models*

In [82]:
pipeline2 = Pipeline(stages=[pipeline, sqlTrans, assembler])

piped_data = pipeline2.fit(obesity).transform(obesity)
piped_data.show(30)

+----+------------------------+----+---+-------------+-----+--------------------+
| Age|Family_History_Indicator|CH2O|FAF|Height_Inches|label|            features|
+----+------------------------+----+---+-------------+-----+--------------------+
|21.0|                     0.0| 2.0|0.0|      63.7794|  1.0|[21.0,0.0,2.0,0.0...|
|21.0|                     0.0| 3.0|3.0|      59.8424|  1.0|[21.0,0.0,3.0,3.0...|
|23.0|                     0.0| 2.0|2.0|       70.866|  1.0|[23.0,0.0,2.0,2.0...|
|27.0|                     1.0| 2.0|2.0|       70.866|  1.0|[27.0,1.0,2.0,2.0...|
|22.0|                     1.0| 2.0|0.0|      70.0786|  1.0|[22.0,1.0,2.0,0.0...|
|29.0|                     1.0| 2.0|0.0|      63.7794|  1.0|[29.0,1.0,2.0,0.0...|
|23.0|                     0.0| 2.0|1.0|       59.055|  1.0|[23.0,0.0,2.0,1.0...|
|22.0|                     1.0| 2.0|3.0|      64.5668|  1.0|[22.0,1.0,2.0,3.0...|
|24.0|                     0.0| 2.0|1.0|      70.0786|  1.0|[24.0,0.0,2.0,1.0...|
|22.0|          

## Model Building

In [83]:
train, test = obesity.randomSplit([0.8,0.2], seed = 1)
print(train.count(), test.count())

1704 407


We have divided our data into a training set and a test set to be able to test which model will perform the best in determining if a given person is considered obese.

### Model 1 - Linear Regression

In [89]:
lr = LinearRegression()
paramGrid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0, 0.01, 0.04, 0.06, 0.1]) \
    .addGrid(lr.elasticNetParam, [0, 0.5, 0.8, 0.9, 1]) \
    .build()
pipeline3 = Pipeline(stages = [pipeline2, lr])

In [90]:
crossval = CrossValidator(estimator = pipeline3,
                          estimatorParamMaps = paramGrid,
                          evaluator = RegressionEvaluator(metricName='rmse'),
                          numFolds=5)

In [91]:
cvModel = crossval.fit(train)

26/04/14 23:33:42 WARN CacheManager: Asked to cache already cached data.
26/04/14 23:33:42 WARN CacheManager: Asked to cache already cached data.
26/04/14 23:33:44 WARN Instrumentation: [a4fac186] regParam is zero, which might cause numerical instability and overfitting.
26/04/14 23:33:46 WARN Instrumentation: [6fb4bd0a] regParam is zero, which might cause numerical instability and overfitting.
26/04/14 23:33:48 WARN Instrumentation: [324f3f2d] regParam is zero, which might cause numerical instability and overfitting.
26/04/14 23:33:49 WARN Instrumentation: [8e2f8f7d] regParam is zero, which might cause numerical instability and overfitting.
26/04/14 23:33:50 WARN Instrumentation: [6ac02a79] regParam is zero, which might cause numerical instability and overfitting.
26/04/14 23:34:14 WARN Instrumentation: [436c06e1] regParam is zero, which might cause numerical instability and overfitting.
26/04/14 23:34:15 WARN Instrumentation: [97e92209] regParam is zero, which might cause numerical i

In [121]:
lr_model = cvModel.bestModel

lr_model = lr_model.stages[-1]

print("Coefficients:", lr_model.coefficients)
print("Intercept:", lr_model.intercept)

Coefficients: [-0.006706496147894034,0.4640953872211685,-0.14063270895775973,0.04798924740721391,0.022305967787936438]
Intercept: -0.6345583471588323


In [118]:
metrics = cvModel.avgMetrics
best_index = np.argmin(metrics)
best_rmse = metrics[best_index]
best_p = paramGrid[best_index].values()

print("Best RMSE:", best_rmse)
print("Best Params:", best_p)

Best RMSE: 0.4536119282122278
Best Params: dict_values([0.0, 0.0])


### First Result

Using Linear Regression, the lowest RMSE was 0.4536119282122278, with the parameter values of 0 and 0. 

### Model 2 - Classification Tree

In [149]:
dt = DecisionTreeRegressor(
    labelCol="label",
    featuresCol="features"
)

pipeline4 = Pipeline(stages = [pipeline2, dt])

In [150]:
paramGrid2 = (ParamGridBuilder()
    .addGrid(dt.maxDepth, [2, 5, 10])
    .addGrid(dt.minInstancesPerNode, [1, 5, 10])
    .build()
)

In [153]:
crossval2 = CrossValidator(
    estimator=pipeline4,
    estimatorParamMaps=paramGrid2,
    evaluator=RegressionEvaluator(metricName='rmse'),
    numFolds=5
)

In [154]:
cvModel2 = crossval2.fit(train)

26/04/15 00:13:56 WARN CacheManager: Asked to cache already cached data.
26/04/15 00:13:56 WARN CacheManager: Asked to cache already cached data.


### Model 2 Results

In [169]:
metrics2 = cvModel2.avgMetrics
best_index2 = np.argmin(metrics2)
best_rmse2 = metrics2[best_index2]
best_p2 = paramGrid2[best_index2].values()

print("Best RMSE:", best_rmse2)
print("Best Params:", best_p2)

Best RMSE: 0.3834369350432957
Best Params: dict_values([10, 10])


The best model produced a RMSE of 0.3834 (lower than the linear regression model's RMSE). The max depth of the model was 10, and the min number of instances per node was also 10.

### Model 3 - Random Forest

In [164]:
rf = RandomForestRegressor(
    featuresCol="features",
    labelCol="label",
    numTrees=100
)

pipeline5 = Pipeline(stages = [pipeline2, rf])

paramGrid3 = (ParamGridBuilder()
    .addGrid(rf.numTrees, [20, 50, 100])
    .addGrid(rf.maxDepth, [3, 5, 10])
    .addGrid(rf.maxBins, [16, 32])
    .build()
)

In [165]:
crossval3 = CrossValidator(
    estimator=pipeline5,
    estimatorParamMaps=paramGrid,
    evaluator=RegressionEvaluator(metricName = 'rmse'),
    numFolds=5
)

In [166]:
cvModel3 = crossval3.fit(train)

### Model 3 Results

In [168]:
metrics3 = cvModel3.avgMetrics
best_index3 = np.argmin(metrics3)
best_rmse3 = metrics3[best_index3]
best_p3 = paramGrid3[best_index3].values()

print("Best RMSE:", best_rmse3)
print("Best Params:", best_p3)

Best RMSE: 0.38700333644812174
Best Params: dict_values([20, 3, 16])


The best RMSE was actually lower than that of the RegressionTree for our given parameter grids. The best model used 20 trees with a max depth of 3, and 16 bins. The max depth on the RegressionTree was 10, showing a clear difference between the two models.

## Model Testing

In [173]:
test_error_1 = RegressionEvaluator().evaluate(cvModel.transform(test))
test_error_2 = RegressionEvaluator().evaluate(cvModel2.transform(test))
test_error_3 = RegressionEvaluator().evaluate(cvModel3.transform(test))
print(test_error_1)
print(test_error_2)
print(test_error_3)

0.44561336180657
0.37294112641933574
0.39150872489208244


When testing our three different models, as with the training set, the Regression Tree Model performed the best. It produced the lowest RMSE compared to the other two models.

I would like to extend this to use Accuracy as a metric for the Regression Tree and Random Forest, as well as include a K-Nearest Neighbor Model, time permitting.

We also could change the parameters in the different parameter grids to check more options given more time and computing power.